# Zero-ablate a layer

Fork a trace, edit one site in the copy, and let the downstream computation rerun, without touching the model.

In [ ]:
import torch
from torch import nn

import torchlens as tl


def tiny_model(seed: int = 0) -> nn.Module:
    """A deterministic three-layer MLP."""

    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Linear(4, 8),
        nn.ReLU(),
        nn.Linear(8, 2),
    )


torch.manual_seed(0)
model = tiny_model(seed=7).eval()
x = torch.randn(2, 4)

Fork the trace and zero the ReLU in the copy; `do` reruns everything downstream.

In [ ]:
clean = tl.trace(model, x, save="all")
ablated = clean.fork().do(tl.label("relu_1_2"), tl.zero_ablate())

The output changed because the ReLU output was zero before the last layer ran.

In [ ]:
print("clean output:", clean.output_ops[-1].out)
print("ablated output:", ablated.output_ops[-1].out)
print(
    "changed:",
    not torch.allclose(clean.output_ops[-1].out, ablated.output_ops[-1].out),
)